## Env

In [5]:
from google.colab import drive
drive.mount('/content/drive')
#Install pakages
project_path = "/content/drive/MyDrive/Sem-6/WorkingArea/Result/NAS_RESULT_v2"
os.chdir(project_path)
print("cwd →", os.getcwd())
%cd $project_path
try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
    data_table.DataTable.max_columns = 50
    data_table.DataTable.max_rows = 150
except ImportError:
    pass


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd → /content/drive/MyDrive/Sem-6/WorkingArea/Result/NAS_RESULT_v2
/content


##copy

In [6]:
import pandas as pd, os

# project_path is already defined in the first cell = ".../WorkingArea/Result/NAS_RESULT_v2"
ROOT = "/content/drive/MyDrive/Sem-6/coding/github/fraud_detection/"   # = root_path in baseline.yaml

SRC = os.path.join(ROOT, "results", "NAS_v2")
DST = project_path
os.makedirs(DST, exist_ok=True)

MODELS = {   # analysis filename : trial-log prefix in NAS_v2
    "timenet.txt": "nas_timesnet_trial_log_WL",
    "lstm.txt":    "nas_lstm_trial_log_WL",
    "tr.txt":      "nas_transformer_trial_log_WL",
    "rf.txt":      "nas_rf_trial_log_WL",
    "xgb.txt":     "nas_xgb_trial_log_WL",
}

for out_name, prefix in MODELS.items():
    parts = []
    for sl in (4, 6, 16):
        p = os.path.join(SRC, f"{prefix}{sl}.csv")
        df = pd.read_csv(p)
        assert (df["seq_length"] == sl).all(), f"{p}: seq_length mismatch"
        assert len(df) == 100, f"{p}: expected 100 rows, got {len(df)}"
        assert df["status"].eq("OK").all(), f"{p}: non-OK rows present"
        parts.append(df)
    alldf = pd.concat(parts, ignore_index=True)
    alldf.to_csv(os.path.join(DST, out_name), sep="\t", index=False)
    print(f"✅ {out_name}: {len(alldf)} rows (3 × 100)")

✅ timenet.txt: 300 rows (3 × 100)
✅ lstm.txt: 300 rows (3 × 100)
✅ tr.txt: 300 rows (3 × 100)
✅ rf.txt: 300 rows (3 × 100)
✅ xgb.txt: 300 rows (3 × 100)


## Check empty records

In [7]:
import pandas as pd
from IPython.display import display

files = {
    "TimesNet": "timenet.txt",
    "LSTM": "lstm.txt",
    "Transformer": "tr.txt",
    "XGBoost": "xgb.txt",
    "RandomForest": "rf.txt",
}

for model_name, file_path in files.items():
    print(f"\n==============================")
    print(f"Checking file: {file_path}")
    print(f"Model name: {model_name}")
    print(f"==============================")

    df = pd.read_csv(file_path, sep="\t")
    df.columns = df.columns.str.strip()
    df["model"] = model_name

    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

    # Check required columns
    required_cols = ["model", "seq_length", "trial_id", "val_f1", "test_f1", "gap_val_test", "is_enqueued"]
    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        print("❌ Missing columns:", missing)
        continue

    # Convert to numeric temporarily for checking
    check_df = df.copy()
    for col in ["seq_length", "trial_id", "val_f1", "test_f1", "gap_val_test"]:
        check_df[col + "_numeric"] = pd.to_numeric(check_df[col], errors="coerce")

    # Show rows with invalid numeric values
    invalid_rows = check_df[
        check_df[
            ["seq_length_numeric", "trial_id_numeric", "val_f1_numeric", "test_f1_numeric", "gap_val_test_numeric"]
        ].isna().any(axis=1)
    ]

    if not invalid_rows.empty:
        print("\n❌ Invalid numeric rows found:")
        display(
            invalid_rows[
                ["seq_length", "trial_id", "val_f1", "test_f1", "gap_val_test", "is_enqueued"]
            ]
        )
    else:
        print("✅ No invalid numeric rows.")

    # Check each group
    for group_key, group_data in df.groupby(["model", "seq_length"], dropna=False):
        model, sl = group_key

        print(f"\nGroup: model={model}, seq_length={sl}, rows={len(group_data)}")

        g = group_data.copy()
        g["val_f1_numeric"] = pd.to_numeric(g["val_f1"], errors="coerce")
        g["test_f1_numeric"] = pd.to_numeric(g["test_f1"], errors="coerce")

        valid_group = g.dropna(subset=["val_f1_numeric", "test_f1_numeric"])

        print("Valid rows:", len(valid_group))

        if valid_group.empty:
            print("❌ This group has ZERO valid rows. Problem rows:")
            display(g[["seq_length", "trial_id", "val_f1", "test_f1", "gap_val_test", "is_enqueued"]])
            continue

        # Debug candidate selection
        max_val = valid_group["val_f1_numeric"].max()
        candidates = valid_group[valid_group["val_f1_numeric"] == max_val]

        print("Max val_f1:", max_val)
        print("Candidates after max val_f1:", len(candidates))

        if candidates.empty:
            print("❌ Candidate became empty after max val_f1.")
            display(valid_group[["seq_length", "trial_id", "val_f1", "test_f1", "gap_val_test", "is_enqueued"]])


Checking file: timenet.txt
Model name: TimesNet
Shape: (300, 29)
Columns: ['date', 'time', 'seq_length', 'trial_id', 'val_f1', 'val_auc', 'val_recall', 'val_precision', 'd_model', 'lr', 'd_ff', 'e_layers', 'top_k', 'num_kernels', 'dropout', 'batch_size', 'patience', 'val_threshold', 'best_test_threshold', 'best_val_so_far', 'best_trial_id', 'test_f1', 'test_auc', 'test_recall', 'test_precision', 'gap_val_test', 'is_enqueued', 'status', 'model']
✅ No invalid numeric rows.

Group: model=TimesNet, seq_length=4, rows=100
Valid rows: 100
Max val_f1: 0.7094
Candidates after max val_f1: 1

Group: model=TimesNet, seq_length=6, rows=100
Valid rows: 100
Max val_f1: 0.7273
Candidates after max val_f1: 1

Group: model=TimesNet, seq_length=16, rows=100
Valid rows: 100
Max val_f1: 0.7561
Candidates after max val_f1: 1

Checking file: lstm.txt
Model name: LSTM
Shape: (300, 27)
Columns: ['date', 'time', 'seq_length', 'trial_id', 'model', 'val_f1', 'val_auc', 'val_recall', 'val_precision', 'lr', 'd_mo

## Base

In [8]:

def format_selected_row_details(row: pd.Series) -> str:
    """
    Convert a selected original row into:
    column:value, column:value, ...
    """
    return ", ".join([f"{col}:{row[col]}" for col in row.index])

# ============================================================
# 1. Configuration
# ============================================================

files = {
    "RandomForest": "rf.txt",
    "TimesNet": "timenet.txt",
    "LSTM": "lstm.txt",
    "Transformer": "tr.txt",
    "XGBoost": "xgb.txt",
}

GROUP_COLS = ("model", "seq_length")
K = 1.0
PERCENTILE = 75
LAMBDA = 1.0
EPS = 0.01   # near-tie band on val_f1 (~1 SE). Fixed a priori — never tune after seeing picks.

# ============================================================
# 2. Helper functions
# ============================================================

def prepare_gap_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds:
    - abs_gap = |val_f1 - test_f1|
    - relative_gap = abs_gap / val_f1
    """
    df = df.copy()

    df["val_f1"] = pd.to_numeric(df["val_f1"], errors="coerce")
    df["test_f1"] = pd.to_numeric(df["test_f1"], errors="coerce")
    df["trial_id"] = pd.to_numeric(df["trial_id"], errors="coerce")
    df["val_auc"] = pd.to_numeric(df["val_auc"], errors="coerce")
    df["seq_length"] = pd.to_numeric(df["seq_length"], errors="coerce")

    df = df.dropna(subset=["val_f1", "test_f1", "trial_id", "seq_length"])

    df["trial_id"] = df["trial_id"].astype(int)
    df["seq_length"] = df["seq_length"].astype(int)

    df["abs_gap"] = (df["val_f1"] - df["test_f1"]).abs()
    df["relative_gap"] = df["abs_gap"] / df["val_f1"]

    return df


def calculate_delta(
    group: pd.DataFrame,
    method: str = "percentile",
    k: float = 1.0,
    percentile: float = 75
) -> float:
    """
    Calculates delta from the abs_gap distribution.

    method="mean_std":
        delta = mean(abs_gap) + k * std(abs_gap)

    method="percentile":
        delta = percentile(abs_gap)
    """
    g = prepare_gap_columns(group)

    if method == "mean_std":
        return g["abs_gap"].mean() + k * g["abs_gap"].std()

    if method == "percentile":
        return g["abs_gap"].quantile(percentile / 100)

    raise ValueError("method must be either 'mean_std' or 'percentile'")


def select_by_validation_rule(group: pd.DataFrame) -> pd.Series:
    """
    NAS baseline rule:

    1) choose highest val_f1
    2) if tie -> highest val_auc
    3) if tie -> lowest trial_id
    """
    g = prepare_gap_columns(group)

    candidates = g[g["val_f1"] == g["val_f1"].max()]

    if len(candidates) > 1:
        candidates = candidates[candidates["val_auc"] == candidates["val_auc"].max()]

    selected = candidates.sort_values("trial_id").iloc[0].copy()
    selected["selection_mode"] = "max_val_f1"
    return selected

def select_by_stability_rule(group: pd.DataFrame, eps: float = EPS) -> pd.Series:
    """
    Headline selection rule (validation-only):
    1) band: trials with val_f1 >= max(val_f1) - eps   (statistical near-ties)
    2) among band -> highest val_auc                    (threshold-free quality)
    3) tie -> smallest architecture                     (Occam)
    4) tie -> lowest trial_id                           (deterministic)
    """
    g = prepare_gap_columns(group)
    band = g[g["val_f1"] >= g["val_f1"].max() - eps].copy()

    if band["val_auc"].notna().any():
        band = band[band["val_auc"] == band["val_auc"].max()]

    if len(band) > 1:
        if "d_model" in band.columns:            # deep models
            band["cx"] = pd.to_numeric(band["d_model"], errors="coerce") * \
                         pd.to_numeric(band["e_layers"], errors="coerce")
        elif "n_estimators" in band.columns:     # RF / XGBoost
            band["cx"] = pd.to_numeric(band["n_estimators"], errors="coerce") * \
                         pd.to_numeric(band["max_depth"], errors="coerce")
        if "cx" in band.columns and band["cx"].notna().any():
            band = band[band["cx"] == band["cx"].min()]

    selected = band.sort_values("trial_id").iloc[0].copy()
    selected["selection_mode"] = f"val_stability_eps{eps}"
    return selected

def select_by_financial_rule(
    group: pd.DataFrame,
    delta_method: str = "percentile",
    k: float = 1.0,
    percentile: float = 75,
    lam: float = 1.0
) -> pd.Series:
    """
    Final fraud / financial selection rule:

    Step 1:
        Calculate delta using either:
        - mean_std
        - percentile

    Step 2:
        Keep stable trials:
            abs_gap <= delta

    Step 3:
        Among stable trials, compute:
            score = test_f1 - lam * relative_gap

    Step 4:
        Select maximum score.

    Tie-breaks:
        1) higher test_f1
        2) higher val_f1
        3) smaller abs_gap
    """
    g = prepare_gap_columns(group)

    delta = calculate_delta(
        g,
        method=delta_method,
        k=k,
        percentile=percentile
    )

    stable = g[g["abs_gap"] <= delta].copy()

    if stable.empty:
        stable = g.copy()
        mode = "fallback_no_stable_trial"
    else:
        mode = f"stable_{delta_method}"

    stable["financial_score"] = (
        stable["test_f1"] - lam * stable["relative_gap"]
    )

    candidates = stable[stable["financial_score"] == stable["financial_score"].max()]

    if len(candidates) > 1:
        candidates = candidates[candidates["test_f1"] == candidates["test_f1"].max()]

    if len(candidates) > 1:
        candidates = candidates[candidates["val_f1"] == candidates["val_f1"].max()]

    if len(candidates) > 1:
        candidates = candidates[candidates["abs_gap"] == candidates["abs_gap"].min()]

    selected = candidates.iloc[0].copy()
    selected["delta"] = delta
    selected["selection_mode"] = mode
    return selected


def build_delta_comparison_table(
    df: pd.DataFrame,
    group_cols=("model", "seq_length"),
    k: float = 1.0,
    percentile: float = 75,
    lam: float = 1.0,
    enqueued_only: bool = False
) -> pd.DataFrame:

    work = df.copy()

    work["is_enqueued"] = (
        work["is_enqueued"]
        .astype(str)
        .str.lower()
        .str.strip()
        .map({"true": True, "false": False})
    )
    if enqueued_only:
        work = work[work["is_enqueued"] == True].copy()   # Manual pool
    else:
        work = work[work["is_enqueued"] == False].copy()  # TPE-only pool (Table 5.1)

    work = prepare_gap_columns(work)

    rows = []

    for group_key, group_data in work.groupby(list(group_cols), dropna=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)

        # old rule: computed ONLY to flag which picks changed (not displayed)
        percentile_row = select_by_financial_rule(
            group_data, delta_method="percentile",
            k=k, percentile=percentile, lam=lam
        )

        # current rule
        stab_row = select_by_stability_rule(group_data)

        row = {col: val for col, val in zip(group_cols, group_key)}

        row["selected_trial_id"]   = int(stab_row["trial_id"])
        row["selected_val_f1"]     = float(stab_row["val_f1"])
        row["selected_val_auc"]    = float(stab_row["val_auc"])
        row["selected_test_f1"]    = float(stab_row["test_f1"])   # report-only
        row["selected_abs_gap"]    = float(stab_row["abs_gap"])   # diagnostic
        row["pick_changed_vs_old"] = row["selected_trial_id"] != int(percentile_row["trial_id"])
        row["selected_details"]    = format_selected_row_details(stab_row)

        rows.append(row)

    return pd.DataFrame(rows)


def load_and_process_all_files(
    files: dict,
    enqueued_only: bool = False
) -> pd.DataFrame:
    """
    Loops over all model files and returns one combined comparison table.
    """
    all_results = []

    for model_name, file_path in files.items():
        df = pd.read_csv(file_path, sep="\t")
        df.columns = df.columns.str.strip()

        # Force consistent model name
        df["model"] = model_name

        comparison_df = build_delta_comparison_table(
            df,
            group_cols=GROUP_COLS,
            k=K,
            percentile=PERCENTILE,
            lam=LAMBDA,
            enqueued_only=enqueued_only
        )

        comparison_df["source_file"] = file_path
        all_results.append(comparison_df)

    return pd.concat(all_results, ignore_index=True)


# ============================================================
# 3. Run analysis
# ============================================================

final_all_trials_df = load_and_process_all_files(
    files,
    enqueued_only=False
)

final_enqueued_trials_df = load_and_process_all_files(
    files,
    enqueued_only=True
)







## Reorder

In [9]:
def reorder_columns(df: pd.DataFrame) -> pd.DataFrame:
    ordered_cols = [
        "model", "seq_length",
        "selected_trial_id", "selected_val_f1", "selected_val_auc",
        "selected_test_f1", "selected_abs_gap",
        "val_rule_trial_id", "val_rule_val_f1", "val_rule_test_f1",
        "percentile_trial_id", "percentile_test_f1",
        "pick_changed_vs_old", "selected_details", "source_file",
    ]
    cols = [c for c in ordered_cols if c in df.columns]
    rest = [c for c in df.columns if c not in cols]
    return df[cols + rest]

    # 🔥 Apply reordering
final_all_trials_df = reorder_columns(final_all_trials_df)
final_enqueued_trials_df = reorder_columns(final_enqueued_trials_df)

## Show best trails for NAS

In [10]:

display(final_all_trials_df)
final_all_trials_df.to_csv("final_all_trials_selection.csv", index=False)


,model,seq_length,selected_trial_id,selected_val_f1,selected_val_auc,selected_test_f1,selected_abs_gap,pick_changed_vs_old,selected_details,source_file
0,RandomForest,4,11,0.6246,0.7740,0.5995,0.0251,True,"date:2026-07-09, time:10:32:26, seq_length:4, ...",rf.txt
1,RandomForest,6,65,0.6584,0.8025,0.6276,0.0308,True,"date:2026-07-09, time:11:35:52, seq_length:6, ...",rf.txt
2,RandomForest,16,34,0.7048,0.8483,0.6718,0.0330,True,"date:2026-07-09, time:14:10:04, seq_length:16,...",rf.txt
3,TimesNet,4,56,0.7045,0.8460,0.6667,0.0378,True,"date:2026-07-08, time:21:20:18, seq_length:4, ...",timenet.txt
4,TimesNet,6,75,0.7206,0.8575,0.7047,0.0159,True,"date:2026-07-09, time:06:20:43, seq_length:6, ...",timenet.txt
5,TimesNet,16,27,0.7480,0.8698,0.7183,0.0297,True,"date:2026-07-09, time:04:10:49, seq_length:16,...",timenet.txt
6,LSTM,4,97,0.7076,0.8463,0.6887,0.0189,True,"date:2026-07-09, time:09:57:13, seq_length:4, ...",lstm.txt
7,LSTM,6,46,0.7276,0.8556,0.6860,0.0416,True,"date:2026-07-09, time:11:58:11, seq_length:6, ...",lstm.txt
8,LSTM,16,41,0.7597,0.8754,0.7218,0.0379,True,"date:2026-07-09, time:13:06:23, seq_length:16,...",lstm.txt
9,Transformer,4,17,0.7055,0.8435,0.6995,0.0060,True,"date:2026-07-09, time:09:19:43, seq_length:4, ...",tr.txt


## Show best trails for Manual seetings

In [11]:

display(final_enqueued_trials_df)

final_enqueued_trials_df.to_csv("final_enqueued_trials_selection.csv", index=False)

,model,seq_length,selected_trial_id,selected_val_f1,selected_val_auc,selected_test_f1,selected_abs_gap,pick_changed_vs_old,selected_details,source_file
0,RandomForest,4,2,0.6241,0.7730,0.5882,0.0359,True,"date:2026-07-09, time:10:32:18, seq_length:4, ...",rf.txt
1,RandomForest,6,3,0.6528,0.7963,0.6299,0.0229,False,"date:2026-07-09, time:11:34:54, seq_length:6, ...",rf.txt
2,RandomForest,16,0,0.7034,0.8488,0.6623,0.0411,True,"date:2026-07-09, time:14:09:30, seq_length:16,...",rf.txt
3,TimesNet,4,2,0.6834,0.8287,0.6793,0.0041,False,"date:2026-07-08, time:17:58:51, seq_length:4, ...",timenet.txt
4,TimesNet,6,1,0.7020,0.8487,0.6843,0.0177,True,"date:2026-07-09, time:01:55:05, seq_length:6, ...",timenet.txt
5,TimesNet,16,1,0.7410,0.8593,0.7035,0.0375,True,"date:2026-07-09, time:01:56:17, seq_length:16,...",timenet.txt
6,LSTM,4,5,0.6992,0.8377,0.6694,0.0298,True,"date:2026-07-09, time:09:10:16, seq_length:4, ...",lstm.txt
7,LSTM,6,5,0.7191,0.8523,0.6878,0.0313,True,"date:2026-07-09, time:11:36:54, seq_length:6, ...",lstm.txt
8,LSTM,16,5,0.7538,0.8709,0.7109,0.0429,True,"date:2026-07-09, time:12:44:36, seq_length:16,...",lstm.txt
9,Transformer,4,3,0.6954,0.8395,0.6765,0.0189,True,"date:2026-07-09, time:09:10:08, seq_length:4, ...",tr.txt


## format the models

In [12]:
def extract_selected_parameters(
    df: pd.DataFrame,
    selected_trials_df: pd.DataFrame,
    model_name: str
):
    """
    Extract parameters for selected trials and format as Python blocks.
    """

    outputs = []

    for _, row in selected_trials_df.iterrows():
        sl = row["seq_length"]
        trial_id = row["selected_trial_id"]

        selected = df[
            (df["trial_id"] == trial_id) &
            (df["seq_length"] == sl)
        ]

        if selected.empty:
            continue

        selected = selected.iloc[0]

        params = selected.to_dict()

        outputs.append((sl, params))

    return outputs

In [13]:
def generate_configs_grouped_by_sl(files, final_df, enqueued_only=False):
    outputs = []

    MODEL_ORDER = ["TimesNet", "LSTM", "Transformer", "XGBoost", "RandomForest"]

    for sl in sorted(final_df["seq_length"].unique()):

        outputs.append(f"\n# ==================================================")
        outputs.append(f"# Base_SL = {sl}")
        outputs.append(f"# ==================================================\n")
        outputs.append(f"Base_SL = {int(sl)}\n")

        selected_for_sl = final_df[final_df["seq_length"] == sl]

        for model_name in MODEL_ORDER:
            if model_name not in files:
                continue

            file_path = files[model_name]

            selected_rows = selected_for_sl[selected_for_sl["model"] == model_name]

            if selected_rows.empty:
                continue

            df = pd.read_csv(file_path, sep="\t")
            df.columns = df.columns.str.strip()
            df["model"] = model_name

            if enqueued_only:
                df["is_enqueued"] = (
                    df["is_enqueued"]
                    .astype(str)
                    .str.lower()
                    .str.strip()
                    .map({"true": True, "false": False})
                )
                df = df[df["is_enqueued"] == True]

            trial_id = int(selected_rows.iloc[0]["selected_trial_id"])

            selected = df[
                (pd.to_numeric(df["trial_id"], errors="coerce") == trial_id) &
                (pd.to_numeric(df["seq_length"], errors="coerce") == int(sl))
            ]

            if selected.empty:
                outputs.append(f"# WARNING: {model_name} SL={sl} trial={trial_id} not found\n")
                continue

            params = selected.iloc[0].to_dict()

            outputs.append(format_single_model_parameters(model_name, params))
            outputs.append("")

    return "\n".join(outputs)


def format_single_model_parameters(model_name, params):
    if model_name == "TimesNet":
        return f'''timesnet_parameters = {{
    "learning_rate": {params.get("learning_rate", params.get("lr"))},
    "d_model":       {params.get("d_model")},
    "d_ff":          {params.get("d_ff")},
    "e_layers":      {params.get("e_layers")},
    "top_k":         {params.get("top_k")},
    "dropout":       {params.get("dropout")},
    "batch_size":    {params.get("batch_size")},
    "patience":      {params.get("patience")},
    "num_kernels":   {params.get("num_kernels")},
}}'''

    if model_name == "LSTM":
        return f'''lstm_parameters = {{
    "learning_rate": {params.get("learning_rate", params.get("lr"))},
    "d_model":       {params.get("d_model")},
    "d_ff":          {params.get("d_ff")},
    "e_layers":      {params.get("e_layers")},
    "dropout":       {params.get("dropout")},
    "batch_size":    {params.get("batch_size")},
    "patience":      {params.get("patience")},
}}'''

    if model_name == "Transformer":
        return f'''transformer_parameters = {{
    "learning_rate": {params.get("learning_rate", params.get("lr"))},
    "d_model":       {params.get("d_model")},
    "n_heads":       {params.get("n_heads")},
    "d_ff":          {params.get("d_ff")},
    "e_layers":      {params.get("e_layers")},
    "dropout":       {params.get("dropout")},
    "batch_size":    {params.get("batch_size")},
    "patience":      {params.get("patience")},
}}'''

    if model_name == "XGBoost":
        return f'''xgb_parameters = {{
    "n_estimators":     {params.get("n_estimators")},
    "max_depth":        {params.get("max_depth")},
    "learning_rate":    {params.get("learning_rate", params.get("lr"))},
    "subsample":        {params.get("subsample")},
    "colsample_bytree": {params.get("colsample_bytree")},
    "min_child_weight": {params.get("min_child_weight")},
    "gamma":            {params.get("gamma")},
    "patience":         {params.get("patience")},
    "eval_metric":      "logloss",
}}'''

    if model_name == "RandomForest":
        return f'''rf_parameters = {{
    "n_estimators":      {params.get("n_estimators")},
    "max_depth":         {params.get("max_depth")},
    "min_samples_split": {params.get("min_samples_split")},
    "min_samples_leaf":  {params.get("min_samples_leaf")},
}}'''

    return f"# Unknown model: {model_name}"

## JSON Output for NAS

In [14]:
configs_normal_by_sl = generate_configs_grouped_by_sl(
    files,
    final_all_trials_df,
    enqueued_only=False
)

print(configs_normal_by_sl)


# ==================================================
# Base_SL = 4
# ==================================================

Base_SL = 4

timesnet_parameters = {
    "learning_rate": 0.0017898065289489,
    "d_model":       128,
    "d_ff":          8,
    "e_layers":      2,
    "top_k":         2,
    "dropout":       0.3193293820708461,
    "batch_size":    16,
    "patience":      5,
    "num_kernels":   6,
}

lstm_parameters = {
    "learning_rate": 0.0017753464688148,
    "d_model":       64,
    "d_ff":          64,
    "e_layers":      2,
    "dropout":       0.2638015575935464,
    "batch_size":    8,
    "patience":      5,
}

transformer_parameters = {
    "learning_rate": 0.0004870572269026,
    "d_model":       16,
    "n_heads":       8,
    "d_ff":          64,
    "e_layers":      3,
    "dropout":       0.01151767389306,
    "batch_size":    8,
    "patience":      2,
}

xgb_parameters = {
    "n_estimators":     64,
    "max_depth":        5,
    "learning_rate":    0.07

## JSON Output for Manual

In [15]:
configs_enqueued_by_sl = generate_configs_grouped_by_sl(
    files,
    final_enqueued_trials_df,
    enqueued_only=True
)

print(configs_enqueued_by_sl)


# ==================================================
# Base_SL = 4
# ==================================================

Base_SL = 4

timesnet_parameters = {
    "learning_rate": 0.0001,
    "d_model":       32,
    "d_ff":          32,
    "e_layers":      2,
    "top_k":         2,
    "dropout":       0.1,
    "batch_size":    8,
    "patience":      3,
    "num_kernels":   6,
}

lstm_parameters = {
    "learning_rate": 0.000936,
    "d_model":       48,
    "d_ff":          4,
    "e_layers":      2,
    "dropout":       0.387,
    "batch_size":    8,
    "patience":      5,
}

transformer_parameters = {
    "learning_rate": 0.001,
    "d_model":       64,
    "n_heads":       8,
    "d_ff":          128,
    "e_layers":      3,
    "dropout":       0.2,
    "batch_size":    8,
    "patience":      3,
}

xgb_parameters = {
    "n_estimators":     300,
    "max_depth":        3,
    "learning_rate":    0.05,
    "subsample":        0.6,
    "colsample_bytree": 0.6,
    "min_child_w